In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from algbench import read_as_pandas
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, HIGHFIGURE, init_params
from tspn_bnb2.misc.plotting import cactus_plot
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()

In [ ]:
map_root_name = {
    "LongestEdgePlusFurthestSite": "LEFP",
    "RandomRoot": "Random",
    "OrderRoot": "CHR",
    "LongestPair": "LE",
}

In [ ]:
def read_row(simplified: bool):
    def read(row):
        instance = AnnotatedInstance.model_validate_json(
            row["parameters"]["args"]["kwargs"]["instance_json"]
        )
        solution = AnnotatedSolution.model_validate_json(row["result"]["solution"])
        return {
            "solution": solution,
            "gap": solution.relative_gap,
            "is_optimal": solution.is_optimal,
            "instance_type": instance.derive_instance_type(),
            "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
            "instance": instance,
            "solve_time": row["result"]["solve_time"],
            "n": instance.num_polygons(),
            "root_strategy": map_root_name[row["parameters"]["args"]["alg_params"]["root"]],
            "instance_simplified": simplified,
        }

    return read


benchmark = read_as_pandas("results_root_strategy", read_row(simplified=True))

print(
    "loaded",
    len(benchmark),
    "runs",
    benchmark["root_strategy"].unique(),
    len(benchmark["instance_name"].unique()),
)

In [ ]:
benchmark_non_simplified = read_as_pandas(
    "results_root_strategy_non_simplified", read_row(simplified=False)
)
print(
    "loaded",
    len(benchmark_non_simplified),
    "runs",
    benchmark_non_simplified["root_strategy"].unique(),
    len(benchmark_non_simplified["instance_name"].unique()),
)

In [ ]:
benchmark = pd.concat(
    [benchmark, benchmark_non_simplified], ignore_index=True, sort=False
).sort_values(by=["root_strategy"])

In [ ]:
# validate that solutions are correct.
n_checks = 0
for _, row in benchmark.iterrows():
    solution = row["solution"]
    if solution is None:
        continue
    same_instance = benchmark[benchmark["instance_name"] == row["instance_name"]]

    assert solution.check_feasibility(row["instance"], eps=0.01)
    for _, other in same_instance.iterrows():
        if other["solution"] is None:
            continue
        check = solution.plausibility_check(other["solution"], eps=0.01)
        assert check["valid"], (
            f"Check failed for {row['instance_name']} {check}"
            f" {other['solution']} {row['root_strategy']}"
        )
        n_checks += 1

print(n_checks, "checks succeeded")

In [ ]:
fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)

sns.boxplot(
    benchmark,
    x="root_strategy",
    y="solve_time",
    hue="instance_type",
    hue_order=["OSM", "tessellation", "random"],
)

xticks = list(ax.get_xticks())
xticklabels = []

for label in ax.get_xticklabels():
    solutions_for_label = benchmark[(benchmark["root_strategy"] == label.get_text())]
    optimal_percentage = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )

    label = label.get_text() + f"\n[{round(optimal_percentage * 100, 2)}\\%]"
    xticklabels.append(label)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)

ax.set_title("lower is better")
ax.set_xlabel("root node strategy")
ax.set_ylabel("runtime [s]")

leg = ax.legend(loc="upper right")
leg.set_title(None)

fig.savefig("../plots/rq1_root_node_selection/runtimes.pdf", bbox_inches="tight")

In [ ]:
from matplotlib.lines import Line2D

fig, axs = plt.subplots(ncols=2, figsize=HIGHFIGURE)

ax = axs[0]
ax2 = axs[1]

colors = {v: f"C{i}" for i, v in enumerate(benchmark["root_strategy"].unique())}

benchmark_solved = benchmark[benchmark["is_optimal"]]

cactus_plot(
    benchmark_solved[benchmark_solved["instance_simplified"]],
    ax=ax,
    instance_column="instance_name",
    strategy_column="root_strategy",
    runtime_column="solve_time",
    flat_line_to=benchmark["solve_time"].max(),
    colors=colors,
)

# Get the existing legend handles/labels (for strategies) before adding non simplified info
handles, labels = ax.get_legend_handles_labels()


cactus_plot(
    benchmark_solved[~benchmark_solved["instance_simplified"]],
    ax=ax,
    instance_column="instance_name",
    strategy_column="root_strategy",
    runtime_column="solve_time",
    flat_line_to=benchmark["solve_time"].max(),
    linestyle="--",
    colors=colors,
)


# Create handles for simplified vs non-simplified
simplified_handles = [
    Line2D([], [], color="black", linestyle="-", label="with preprocessing"),
    Line2D([], [], color="black", linestyle="--", label="no preprocessing"),
]

# Combine legends: first strategies, then simplified info
ax.legend(
    handles=handles + simplified_handles,
    labels=labels + [h.get_label() for h in simplified_handles],
)


"""
sns.violinplot(
    benchmark,
    x="instance_simplified",
    y="gap",
    hue="root_strategy",
    hue_order=colors.keys(),
    ax=ax2,
)

ax2.legend().remove()

ax2.set_xlabel("")
ax2.set_xticks([False, True])
ax2.set_xticklabels(["no preprocessing", "with preprocessing"])
"""
"""
sns.violinplot(
    data=benchmark,
    x="root_strategy",           # x-axis is root strategies
    y="gap",
    inner="quart",
    fill=False,
    hue="instance_simplified",   # split by preprocessing
    palette=["#d95f02", "#1b9e77"],  # set colors for preprocessing
    split=True,                  # only works if hue has exactly 2 levels
    ax=ax2
)

ax2.set_xlabel("")
ax2.set_ylabel("Gap")
#ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)
ax2.legend(title="Preprocessing")
"""

bm = benchmark[~benchmark["is_optimal"]]
bm["instance_simplified_hr"] = bm["instance_simplified"].apply(
    lambda x: "with preprocessing" if x is True else "no preprocessing"
)
# bm["instance_simplified_hr"] = bm["instance_simplified"]

sns.boxplot(
    data=bm,
    x="root_strategy",
    y="gap",
    hue="instance_simplified_hr",
    palette=["C4", "C5"],
    dodge=True,
    ax=ax2,
)
ax2.set_xlabel("")
ax2.set_ylabel("gap")
ax2.legend()

xticks = list(ax2.get_xticks())
xticklabels = []


def to_percentage(x):
    return round(x * 100, 1)


for label in ax2.get_xticklabels():
    percentages = {}
    for simplified in [False, True]:
        solutions_for_label = benchmark[
            (benchmark["root_strategy"] == label.get_text())
            & (benchmark["instance_simplified"] == simplified)
        ]
        if len(solutions_for_label) == 0:
            continue

        percentages[simplified] = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
            solutions_for_label
        )

    # if percentages.get(False) == percentages[True]:
    #    del percentages[False]

    if len(percentages) == 1:
        label = label.get_text() + f"\n[{to_percentage(percentages[True])}\\%]"
    else:
        label = (
            label.get_text()
            + f"\n[{to_percentage(percentages[True])}\\%,{to_percentage(percentages[False])}\\%]"
        )
    xticklabels.append(label)

ax2.set_xticks(xticks)
ax2.set_xticklabels(xticklabels, rotation=40)


ax.set_title("higher is better")
ax2.set_title("lower is better")
fig.tight_layout()

fig.savefig("test.pdf")